In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

In [2]:
from translate_infer import *

/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pdf_processor import PDFProcessor
from section_parser import SectionHierarchicalParser
from section_merger import SectionMerger
from base_processor import BaseProcessor

In [4]:
model = InferEngine()

/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
pdf = PDFProcessor()
section_parser = SectionHierarchicalParser()
section_merger = SectionMerger()

In [59]:
item = pdf.process('/raid/phundh/translateFeature/pdf/2019_KH-UBND_613002.pdf')

In [8]:
def batched(iterable, n):
    l = len(iterable)
    for ndx in range(0, l, n):
        yield iterable[ndx:min(ndx + n, l)]

In [60]:
# for tables in batched(list(item['tables'].keys()), 4):
#     table_batch = [item['tables'][key] for key in tables]
#     table_output = model.inference(table_batch)

#     # Replace summarized tables to table placeholder (<table 0>, <table 1>,...) in text
#     for key, output in zip(tables, table_output):
#         for ind, section in enumerate(item['sections']):
#             item['sections'][ind] = section.replace(f'<{key}>', output)

item = item['sections']

In [57]:
item

{'value': '',
 'child': [{'value': '\n \n <SECTION> \n ỦY BAN NHÂN DÂN\n I. MỤC TIÊU \n \n 1. Mục tiêu chung: Tăng cường công tác quản lý và nâng cao năng lực, nhận thức trong việc quản lý, sử dụng hóa chất, chế phẩm diệt côn trùng, diệt khuẩn dùng trong lĩnh vực gia dụng và y tế. \n 2. Mục tiêu cụ thể \n \n a) Tăng cường năng lực quản lý hóa chất, chế phẩm từ tuyến tỉnh đến tuyến xã. \n b) Tăng cường công tác thanh tra, kiểm tra về quản lý, sản xuất, mua bán và sử dụng hóa chất, chế phẩm. \n c) Nâng cao nhận thức cho các cơ sở y tế và cộng đồng về quản lý, sử dụng an toàn hóa chất, chế phẩm. \n d) Tăng cường ứng dụng công nghệ thông tin, nghiên cứu khoa học và hợp tác quốc tế trong quản lý hóa chất, chế phẩm. \n 3. Các chỉ tiêu cụ thể\n <table_0>',
   'child': [],
   'len': 163},
  {'value': '\n \n II. GIẢI PHÁP VÀ CÁC NHIỆM VỤ CHỦ YẾU \n \n 1. Công tác chỉ đạo, điều hành \n \n   - Kịp thời rà soát và đề xuất sửa đổi, bổ sung (nếu có) văn bản quy phạm pháp luật về quản lý hóa chất, ch

In [61]:
item = section_parser.section_parsing(item, max_level=300)

In [62]:
item = {'value': '', 'child': item}

In [ ]:
def get_depth(item):
    if isinstance(item, str) or (isinstance(item, dict) and len(item['child']) == 0):
        return 1
    return 1 + max(get_depth(child) for child in item['child'])

In [ ]:
current_target_level = min(get_depth(item) - 1, 300)

In [ ]:
current_target_level

In [36]:
while current_target_level >= 0:
    item = section_merger.merge(item, current_target_level, max_length=512)
    current_target_level -= 1

In [37]:
text_value = [doc['value'] for doc in item['child']]

In [38]:
text_value

['\n \n <SECTION> \n ỦY BAN NHÂN DÂN\n I. MỤC TIÊU \n \n 1. Mục tiêu chung: Tăng cường công tác quản lý và nâng cao năng lực, nhận thức trong việc quản lý, sử dụng hóa chất, chế phẩm diệt côn trùng, diệt khuẩn dùng trong lĩnh vực gia dụng và y tế. \n 2. Mục tiêu cụ thể \n \n a) Tăng cường năng lực quản lý hóa chất, chế phẩm từ tuyến tỉnh đến tuyến xã. \n b) Tăng cường công tác thanh tra, kiểm tra về quản lý, sản xuất, mua bán và sử dụng hóa chất, chế phẩm. \n c) Nâng cao nhận thức cho các cơ sở y tế và cộng đồng về quản lý, sử dụng an toàn hóa chất, chế phẩm. \n d) Tăng cường ứng dụng công nghệ thông tin, nghiên cứu khoa học và hợp tác quốc tế trong quản lý hóa chất, chế phẩm. \n 3. Các chỉ tiêu cụ thể\n <table_0>',
 '\n \n II. GIẢI PHÁP VÀ CÁC NHIỆM VỤ CHỦ YẾU \n \n 1. Công tác chỉ đạo, điều hành \n \n   - Kịp thời rà soát và đề xuất sửa đổi, bổ sung (nếu có) văn bản quy phạm pháp luật về quản lý hóa chất, chế phẩm diệt côn trùng, diệt khuẩn dùng trong lĩnh vực gia dụng và y tế; cập n

In [50]:
%%time

text_output = model.inference(text_value,en_vi=True)

CPU times: user 6.56 s, sys: 17.7 ms, total: 6.58 s
Wall time: 6.51 s


In [54]:
def add_newline_before_headings(text):
    # Biểu thức regex nhận diện các tiêu đề dạng số hoặc chữ cái, ví dụ: "1.", "2.", "III.", "Part I"
    pattern = r'(\b\d+\.\s|\b[A-Z]+\.\s|Part\s+[A-Z]+|\b[a-z]\)\s|\b[a-z]\/\s)'
    
    # Thêm newline trước các đề mục tìm thấy
    new_text = re.sub(pattern, r'\n\n\1', text)
    
    return new_text.strip()  # Xóa khoảng trắng không cần thiết ở đầu và cuối

In [55]:
text_output[1]

'vi: 1. Directing and operating: Promptly reviewing and proposing amendments and supplements (if any) to legislative documents on management of insecticidal and germicidal chemicals and preparations for household and medical use; updating technical guidelines reviewed to propose amendments or promulgation of new chemical and preparation groups (if any); requesting the Ministry of Health in writing or assigning additional functions of chemical and preparation testing to provincial Centers for Drug, Cosmetic and Food Inspection to serve the quality inspection and assurance of chemicals and preparations. - Updating the list of chemicals banned or restricted from use in Vietnam to disseminate and propagate to enterprises, management agencies and people for compliance in accordance with law. - Guiding localities and relevant units to implement administrative procedures on chemicals and preparations as decentralized. * To implement: The Department of Health shall assume the prime responsibil

In [56]:
import re

for inp,oup in zip(text_value,text_output):
    print("=============")
    print(inp)
    print("#############")
    txt = oup.replace(' - ',' \n- ')
    txt = add_newline_before_headings(txt)
    print(txt)
    print("=============")


 
 <SECTION> 
 ỦY BAN NHÂN DÂN
 I. MỤC TIÊU 
 
 1. Mục tiêu chung: Tăng cường công tác quản lý và nâng cao năng lực, nhận thức trong việc quản lý, sử dụng hóa chất, chế phẩm diệt côn trùng, diệt khuẩn dùng trong lĩnh vực gia dụng và y tế. 
 2. Mục tiêu cụ thể 
 
 a) Tăng cường năng lực quản lý hóa chất, chế phẩm từ tuyến tỉnh đến tuyến xã. 
 b) Tăng cường công tác thanh tra, kiểm tra về quản lý, sản xuất, mua bán và sử dụng hóa chất, chế phẩm. 
 c) Nâng cao nhận thức cho các cơ sở y tế và cộng đồng về quản lý, sử dụng an toàn hóa chất, chế phẩm. 
 d) Tăng cường ứng dụng công nghệ thông tin, nghiên cứu khoa học và hợp tác quốc tế trong quản lý hóa chất, chế phẩm. 
 3. Các chỉ tiêu cụ thể
 <table_0>
#############
vi: 

1. General objectives: To enhance the management of, and raise the capacity and awareness of, the management and use of insecticidal and germicidal chemicals and preparations for domestic and medical use. 

2. Specific objectives a/To enhance the capacity of management of

In [49]:
text_output[2]

'en: 3. Enhance chemical and preparation management capacity: Provide training in legislative documents, chemical and preparation management and safety for health officials and officials of socio-political organizations at all levels. Provide training courses in chemical and preparation inspection for inspectors at Departments of Health. Provide training in legislative documents and chemical and preparation safety for producers, sellers and users; methods for provision of level 4 online public services on chemicals and preparations. Organize field trips to learn and exchange experience in chemical and preparation management (if any).'